## 1. Import and helper

In [2]:
import pandas as pd
import sys
sys.path.append("../src")
from prepare_data import DATASETS
from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from cleanlab.filter import find_label_issues

def load(ds_name, cond):
    return pd.read_csv(f"../data/messy/{ds_name}/{cond}/train.csv")

def rule_violations(series, lo, hi):
    bad = series < float("-inf")
    if lo is not None:
        bad = bad | (series < lo)
    if hi is not None:
        bad = bad | (series > hi)
    return bad

def flags_rules(df, cols, bounds):
    flags = pd.DataFrame(False, index=df.index, columns=cols)
    for col in cols:
        lo, hi = bounds.get(col, (None, None))
        flags[col] = rule_violations(df[col], lo, hi)
    return flags

def flags_iqr(df, cols):
    flags = pd.DataFrame(False, index=df.index, columns=cols)
    for col in cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        flags[col] = (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)
    return flags

def flags_iforest(df, cols):
    model = IsolationForest(contamination="auto", random_state=42)
    pred = model.fit(df[cols]).predict(df[cols])
    return pd.Series(pred == -1, index=df.index)

c:\Users\ARYA\Documents\Arya\Projects\POLIMI-GSOM-Projects\04_Thesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
for ds_name, info in DATASETS.items():
    clean = load(ds_name, "clean")
    target = info["target"]
    cols = info["continuous"]
    pos_total = clean[target].sum()

    methods = {
        "IQR": flags_iqr(clean, cols).any(axis=1),
        "IsolationForest": flags_iforest(clean, cols),
        "Validity rules": flags_rules(clean, cols, info["bounds"]).any(axis=1),
    }

    print(f"\n{ds_name.upper()}  (positive rate overall: {clean[target].mean():.1%})")
    print(f"  {'method':<16} {'rows flagged':>13} {'pos. rate flagged':>18} {'positives lost':>15}")
    for name, flagged in methods.items():
        n_flagged = flagged.sum()
        pos_rate = clean.loc[flagged, target].mean() if n_flagged > 0 else 0
        pos_lost = clean.loc[flagged, target].sum() / pos_total
        print(f"  {name:<16} {n_flagged / len(clean):>13.1%} {pos_rate:>18.1%} {pos_lost:>15.1%}")


BANK_MARKETING  (positive rate overall: 11.3%)
  method            rows flagged  pos. rate flagged  positives lost
  IQR                      20.5%              21.4%           39.0%
  IsolationForest          20.8%              30.2%           55.7%
  Validity rules            0.0%               0.0%            0.0%

ONLINE_SHOPPERS  (positive rate overall: 15.5%)
  method            rows flagged  pos. rate flagged  positives lost
  IQR                      57.7%              24.3%           90.6%
  IsolationForest           9.4%              26.6%           16.1%
  Validity rules            0.0%               0.0%            0.0%

CREDIT_CARD  (positive rate overall: 22.1%)
  method            rows flagged  pos. rate flagged  positives lost
  IQR                      35.0%              15.3%           24.1%
  IsolationForest           8.6%              15.4%            6.0%
  Validity rules            0.0%               0.0%            0.0%


In [4]:
for ds_name, info in DATASETS.items():
    clean = load(ds_name, "clean")
    messy = load(ds_name, "outlier_severe")
    cols = info["continuous"]

    injected = messy[cols] != clean[cols]            # per value
    injected_rows = injected.any(axis=1)             # per row

    print(f"\n{ds_name.upper()}")
    print(f"  {'method':<16} {'level':<6} {'caught':>8} {'correct':>8}")

    for name, flags in [("IQR", flags_iqr(messy, cols)),
                        ("Validity rules", flags_rules(messy, cols, info["bounds"]))]:
        hit = (flags & injected).sum().sum()
        caught = hit / injected.sum().sum()
        correct = hit / flags.sum().sum() if flags.sum().sum() > 0 else 0
        print(f"  {name:<16} {'value':<6} {caught:>8.1%} {correct:>8.1%}")

    flagged_rows = flags_iforest(messy, cols)
    hit = (flagged_rows & injected_rows).sum()
    print(f"  {'IsolationForest':<16} {'row':<6} {hit / injected_rows.sum():>8.1%} "
          f"{hit / flagged_rows.sum():>8.1%}")


BANK_MARKETING
  method           level    caught  correct
  IQR              value     49.3%    87.8%
  Validity rules   value     16.6%   100.0%
  IsolationForest  row       21.4%    96.3%

ONLINE_SHOPPERS
  method           level    caught  correct
  IQR              value    100.0%    66.5%
  Validity rules   value     45.1%   100.0%
  IsolationForest  row       10.9%    99.1%

CREDIT_CARD
  method           level    caught  correct
  IQR              value     94.5%    76.2%
  Validity rules   value     28.3%   100.0%
  IsolationForest  row        5.4%    96.3%


In [6]:
def cleanlab_flags(df, target, frac_noise=1.0, filter_by="prune_by_noise_rate", balanced=False):
    X = pd.get_dummies(df.drop(columns=[target]), dtype=float)
    y = df[target].values
    if balanced:
        model = HistGradientBoostingClassifier(random_state=42, class_weight="balanced")
    else:
        model = HistGradientBoostingClassifier(random_state=42)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    probs = cross_val_predict(model, X, y, cv=cv, method="predict_proba")
    issues = find_label_issues(labels=y, pred_probs=probs,
                               frac_noise=frac_noise, filter_by=filter_by)
    return pd.Series(issues, index=df.index)

CANDIDATES = {
    "C0 default":         {"frac_noise": 1.0, "filter_by": "prune_by_noise_rate", "balanced": False},
    "C1 frac_noise=0.5":  {"frac_noise": 0.5, "filter_by": "prune_by_noise_rate", "balanced": False},
    "C2 balanced":        {"frac_noise": 1.0, "filter_by": "prune_by_noise_rate", "balanced": True},
    "C3 balanced+both":   {"frac_noise": 1.0, "filter_by": "both",                "balanced": True},
}

for cand_name, settings in CANDIDATES.items():
    print(f"\n{'=' * 70}\n{cand_name}\n{'=' * 70}")
    print(f"  {'dataset':<16} {'pos lost (clean)':>17} {'rows rem. (clean)':>18} "
          f"{'resid. noise':>13} {'pos rate after':>15} {'true':>6}")

    for ds_name, info in DATASETS.items():
        target = info["target"]
        clean = load(ds_name, "clean")
        noisy = load(ds_name, "noise_severe")
        flipped = noisy[target] != clean[target]

        removed_clean = cleanlab_flags(clean, target, **settings)
        removed_noisy = cleanlab_flags(noisy, target, **settings)

        pos_lost = clean.loc[removed_clean, target].sum() / clean[target].sum()
        rows_removed = removed_clean.mean()
        kept = ~removed_noisy
        resid_noise = (flipped & kept).sum() / kept.sum()
        pos_after = noisy.loc[kept, target].mean()

        print(f"  {ds_name:<16} {pos_lost:>17.1%} {rows_removed:>18.1%} "
              f"{resid_noise:>13.1%} {pos_after:>15.1%} {clean[target].mean():>6.1%}")


C0 default
  dataset           pos lost (clean)  rows rem. (clean)  resid. noise  pos rate after   true
  bank_marketing               40.9%               6.2%          5.6%            9.9%  11.3%
  online_shoppers              14.5%               5.6%          5.3%           16.9%  15.5%
  credit_card                  38.1%              12.3%          8.4%           18.5%  22.1%

C1 frac_noise=0.5
  dataset           pos lost (clean)  rows rem. (clean)  resid. noise  pos rate after   true
  bank_marketing               20.5%               3.9%         13.1%           19.3%  11.3%
  online_shoppers               7.3%               2.8%         12.5%           24.7%  15.5%
  credit_card                  19.1%               8.1%         13.6%           27.5%  22.1%

C2 balanced
  dataset           pos lost (clean)  rows rem. (clean)  resid. noise  pos rate after   true
  bank_marketing               28.9%              14.5%          6.1%           13.1%  11.3%
  online_shoppers         